## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [1]:
# Import Required Libraries.
import os
import sys
import json
import random
from glob import glob

import pandas as pd

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.data.download_data import main as downloadData
from src.baseline.notebook_helpers import get_next_run_dir, load_combined_gt, resolve_local_flights
from src.baseline.notebook_helpers import resolve_local_flights

# Confirm Project Root.
log(LogLevel.INFO, "baseline:imports", "Project Root Set: %s", PROJECT_ROOT)

<INFO ><baseline:imports> Project Root Set: /home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid



---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [2]:
# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/selected_flights.txt"
DATASET_PARTS = ["part1", "part2", "part3"]

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = "outputs/results/baseline/"
run_name = get_next_run_dir(RESULTS_DIR)
RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")

# Evaluation Filtering.
MIN_SCORE = 0.0
MIN_TRACK_LEN = 0

# Evaluation Output Folders.
EVAL_GT_DIR      = "outputs/evaluation/baseline/gt"
EVAL_RESULT_DIR  = "outputs/evaluation/baseline/result"
EVAL_SUMMARY_DIR = "outputs/evaluation/baseline/summaries"

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Path  : {DATASET_PATH}")
print(f"  Dataset Parts : {DATASET_PARTS}")
print(f"  Run Name      : {run_name}")
print(f"  Results Path  : {RESULTS_PATH}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

<INFO ><baseline:config> Configuration Loaded.
  Dataset Path  : configs/datasets/selected_flights.txt
  Dataset Parts : ['part1', 'part2', 'part3']
  Run Name      : run0
  Results Path  : outputs/results/baseline/run0/result.json
  Min Score     : 0.0
  Min Track Len : 0


In [ ]:
# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)


---

<br/>

## **Data Preparation and Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [3]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, PROJECT_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "baseline:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "baseline:preprocessing", "  ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, PROJECT_ROOT)
log(LogLevel.INFO, "baseline:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

<INFO ><notebook_helpers:resolve_flights> 9 Flights on Disk. 0 Not Found. (Run the Download Cell to Fetch Missing Flights).

<INFO ><baseline:preprocessing> Flights Ready for Inference: 9
  [part1]  part100bb96a5a68f4fa5bc5c5dc66ce314d2  (93 images)
  [part1]  part1829d263844ed4387b374fe9db9a510e4  (102 images)
  [part1]  part143a99be53c3a4176a8f7d9e10f8bd29c  (109 images)
  [part1]  part14d196dd04a23411193d525b2356a1187  (109 images)
  [part1]  part101a7a84b9226468bbb7f88d483c3e996  (121 images)
<DEBUG><baseline:preprocessing>   ... and 4 more.

<INFO ><notebook_helpers:load_gt> 9 kept / 4154 total.
<INFO ><baseline:preprocessing> Flights in Filtered Ground Truth: 9



---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

Run `seg_test.py` across each dataset part sequentially and save the results to the configured output directories.

> ⚠️ **This cell requires a CUDA GPU and will take several hours to complete.** Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.

In [ ]:
# Import Data Downloader Libraries.
from src.baseline.seg_test import get_next_run_dir

# Run Inference On Each Dataset Part.
for i, (image_dir, results_dir) in enumerate(zip(IMAGE_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Auto-Increment Run Folder so Previous Results are Never Overwritten.
    run_name = get_next_run_dir(results_dir)

    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:inference", "Running Inference on %s...", part_name)
    print(f"  Images  : {image_dir}")
    print(f"  Results : {results_dir}")
    print(f"  Run Folder : {run_name}")
    print(f"{'-'*58}")

    # Ensure Results Directory Exists. 
    os.makedirs(results_dir, exist_ok=True)

    # Set the Variables to Correct Paths.
    env = os.environ.copy()
    env["TEST_DATASET_PATH"] = image_dir
    env["INFERENCE_OUTPUT_PATH"] = results_dir
    env["DATASET_ENV"] = run_name
    env["MIN_SCORE"] = str(MIN_SCORE)
    env["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
    
    # Run the Basline Model Inference.
    result = subprocess.run(
        [sys.executable, "-m", "src.baseline.seg_test"],
        env=env,
        capture_output=False
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:inference", "%s Inference Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Inference Failed with Return Code %s.\n", part_name, result.returncode)

### **Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Import Data Downloader Libraries.
from src.baseline.seg_test import get_last_run

# Ensure the Summaries Directory Exists. 
os.makedirs(SUMMARIES_DIR, exist_ok=True)

# Run Metrics for Each Part.
for i, (dataset_dir, results_dir) in enumerate(zip(DATASET_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Automatically Find Latest Run.
    run_name = get_last_run(results_dir)
    if run_name is None:
        log(LogLevel.WARNING, "baseline:evaluation", "No Run Folders Found in %s - Skipping.", results_dir)
        continue

    # Set Results Folder Name.
    results_file_dir = os.path.join(results_dir, run_name)
    
    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:evaluation", "Evaluating %s...", part_name)
    print(f"  Ground Truth : {dataset_dir}")
    print(f"  Results      : {results_file_dir}")

    # Check Results Exist Before Evaluating.
    result_files = (
        glob(os.path.join(results_file_dir, "*.json")) +
        glob(os.path.join(results_file_dir, "*.csv"))
    )
    if not result_files:
        log(LogLevel.WARNING, "baseline:evaluation", "No Result Files Found in %s - Skipping.", results_file_dir)
        log(LogLevel.WARNING, "baseline:evaluation", "Run Inference Module First.")
        continue

    # Save Results for Each Part Separately.
    part_summaries_dir = os.path.join(SUMMARIES_DIR, part_name)
    os.makedirs(part_summaries_dir, exist_ok=True)

    # Run the Metrics Calculating Script. 
    result = subprocess.run(
        [
            sys.executable, METRICS_SCRIPT,
            "--dataset-folder", dataset_dir,
            "--results-folder", results_file_dir,
            "--summaries-folder", part_summaries_dir,
            "--min-score", str(MIN_SCORE),
            "--min-track-len", str(MIN_TRACK_LEN),
            "--enc-max-range", "2000",
        ],
        capture_output=True,
        text=True
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:evaluation", "%s Metrics Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Metrics Failed with Error:\nSTDOUT:\n%s\nSTDERR:\n%s\n", 
            part_name, result.stdout, result.stderr)

### **Aggregate Results**

Load the per-part summary JSON files and aggregate them into a single set of benchmark statistics across the full custom dataset.

In [ ]:
# Initialise an Empty List to Store Summaries from All Parts.
all_summaries = []

# Load All Summary JSON Files Across All Parts.
for i in range(1, 4):
    # Construct the Directory Path for the Current Part.
    part_summaries_dir = os.path.join(SUMMARIES_DIR, f"Part {i}")

    # Get a List of All JSON Files in the Current Part's Directory.
    summary_files = glob(os.path.join(part_summaries_dir, "*summary*.json"))

    # Loop through Each Summary File Found.
    for f in summary_files:
        with open(f) as jf:
            summary = json.load(jf)
            summary["part"] = f"part{i}"
            all_summaries.append(summary)

# Check that the Summary Files Exist. 
if not all_summaries:
    log(LogLevel.ERROR, "baseline:aggregation", "No Summary Files Found! Run Steps 1 and 2 First.")
else:
    log(LogLevel.INFO, "baseline:aggregation", "Loaded %d Summary File(s) Across All Parts.\n", len(all_summaries))

    # Store the Records Togther.
    records = []

    # Aggregate Key Metrics Across all Parts.
    for s in all_summaries:
        # Grab Whatever Range Key Exists in the Tracking Encounters Dict.
        tracking_encounters = s.get("Tracking", {}).get("Encounters", {})
        range_key = next(iter(tracking_encounters), None)

        # Append Results.
        records.append({
            "Part"              : s["part"],
            "Min Score"         : s.get("min_det_score", MIN_SCORE),
            "FPPI"              : s.get("fppi"),
            "AFDR"              : s.get("fl_dr_in_range"),
            "HFAR"              : s.get("far"),
            "Detected Aircraft" : s.get("det_aircraft_in_range"),
            "Total Aircraft"    : s.get("tot_aircraft_in_range"),
            "EDR (All)"   : tracking_encounters.get(range_key, {}).get("All", {}).get("dr")           if range_key else None,
            "EDR (Above)" : tracking_encounters.get(range_key, {}).get("Above Horizon", {}).get("dr") if range_key else None,
            "EDR (Below)" : tracking_encounters.get(range_key, {}).get("Below Horizon", {}).get("dr") if range_key else None,
        })

    # Display the Results for Each Part of the Dataset. 
    df_parts = pd.DataFrame(records)
    log(LogLevel.INFO, "baseline:aggregation", "Per-Part Results:")
    display(df_parts)

### **Baseline Benchmark Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide an overall baseline performance for the full hard dataset. Additionally, statistics for each individual subset of the hard dataset are also displayed for more detailed analysis.

In [ ]:
# Check that the Summary Files Exist. 
if not all_summaries:
    log(LogLevel.ERROR, "baseline:summary", "No Summary Files Found! Run Steps 1, 2 and 3 First.")
else:
    # # Aggregate Metrics Across All Parts.
    total_detected   = df_parts["Detected Aircraft"].sum()
    total_aircraft   = df_parts["Total Aircraft"].sum()
    mean_fppi        = df_parts["FPPI"].mean()
    mean_hfar        = df_parts["HFAR"].mean()
    overall_afdr     = total_detected / total_aircraft if total_aircraft > 0 else 0
    mean_edr_all     = df_parts["EDR (All)"].mean()
    mean_edr_above   = df_parts["EDR (Above)"].mean()
    mean_edr_below   = df_parts["EDR (Below)"].mean()

    # Rank Metrics Against Thresholds and Display Results.
    if mean_fppi <= 0.0005:
        ranked_fppi = log(LogLevel.INFO, "baseline:summary", "FPPI Within Acceptable Limit (≤ 0.0005).")
    else:
        ranked_fppi = log(LogLevel.WARNING, "baseline:summary", f"FPPI Exceeds Limit! Actual: {mean_fppi:.6f} (≤ 0.0005).")
    if mean_hfar <= 0.5:
        ranked_hfar = log(LogLevel.INFO, "baseline:summary", "HFAR Within Acceptable Limit (≤ 0.5).")
    else:
        ranked_hfar = log(LogLevel.WARNING, "baseline:summary", f"HFAR Exceeds Limit! Actual: {mean_hfar:.4f} (≤ 0.5).")

    # Build a Summary DataFrame for Display and CSV Export.
    summary_df = pd.DataFrame([
        {"Metric": "AFDR (Airborne Flight Detection Rate)",  "Value": f"{overall_afdr:.4f}",  "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",       "Value": f"{mean_fppi:.6f}",     "Status": ranked_fppi},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",     "Value": f"{mean_hfar:.4f}",     "Status": ranked_hfar},
        {"Metric": "EDR - All Scenarios",                    "Value": f"{mean_edr_all:.4f}",  "Status": ""},
        {"Metric": "EDR - Above Horizon",                    "Value": f"{mean_edr_above:.4f}","Status": ""},
        {"Metric": "EDR - Below Horizon",                    "Value": f"{mean_edr_below:.4f}","Status": ""},
        {"Metric": "Total Aircraft Detected",                "Value": f"{int(total_detected)}","Status": ""},
        {"Metric": "Total Aircraft in Dataset",              "Value": f"{int(total_aircraft)}","Status": ""},
    ])

    # Print Summary to Console.
    print("\n" + "="*60)
    print(" BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY")
    print(" Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)")
    print("="*60)
    display(summary_df.set_index("Metric"))

    # Save Aggregated Summary to CSV.
    output_path = os.path.join(SUMMARIES_DIR, "baseline_benchmark_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print()
    log(LogLevel.INFO, "baseline:summary", "Summary saved to: %s", output_path)


---

<br/>

## **Hard Dataset Results**

The table below reports baseline performance broken down by scenario category across the custom
hard dataset (1,618 flights). Flights were selected from the full AOT dataset using configurable
filter criteria to isolate the most challenging sequences, providing a more targeted benchmark
than the full competition dataset.

| Scenario                     | Flights | FPPI | AFDR | HFAR | EDR |
|------------------------------|---------|------|------|------|-----|
| **Overall (All Criteria)**   |**1,618**| —    | —    | —    | —   |
| Small Object (bbox ≤ 36 px²) | 662     | —    | —    | —    | —   |
| Below Horizon                | 1,053   | —    | —    | —    | —   |
| Multi-Object                 | 627     | —    | —    | —    | —   |
| No Visible Object            | 354     | —    | —    | —    | —   |

*Results evaluated at a minimum detection confidence of 0.985 and a minimum track length of 30 frames (≈ 3 seconds at 10 fps).*

---